# Classification and Evaluation Metrics

<a target="_blank" href="https://colab.research.google.com/github/imamitjain/notebooks/blob/main/02-ml-fundamentals/02_classification_and_metrics.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Objective:** Train classifiers (logistic regression, decision trees, random forests) and evaluate them with accuracy, precision, recall, F1, ROC-AUC, and confusion matrices.

**Prerequisites:** Regression basics (notebook 01)

In [ ]:
import sys

if "google.colab" in sys.modules:
    %pip install -q numpy pandas matplotlib scikit-learn


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, ConfusionMatrixDisplay,
                             classification_report, roc_curve, roc_auc_score)
from sklearn.datasets import make_classification

## 1. Generate and Split Data

In [ ]:
X, y = make_classification(n_samples=500, n_features=20, n_informative=10,
                           n_redundant=5, random_state=42)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training: {X_train.shape[0]} samples")
print(f"Test:     {X_test.shape[0]} samples")
print(f"Class distribution: {np.bincount(y_train)}")

## 2. Logistic Regression

In [ ]:
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_scaled, y_train)
y_pred_lr = lr.predict(X_test_scaled)

print("Logistic Regression Results:")
print(f"  Accuracy:  {accuracy_score(y_test, y_pred_lr):.4f}")
print(f"  Precision: {precision_score(y_test, y_pred_lr):.4f}")
print(f"  Recall:    {recall_score(y_test, y_pred_lr):.4f}")
print(f"  F1:        {f1_score(y_test, y_pred_lr):.4f}")

## 3. Decision Tree and Random Forest

In [ ]:
dt = DecisionTreeClassifier(max_depth=5, random_state=42)
dt.fit(X_train_scaled, y_train)
y_pred_dt = dt.predict(X_test_scaled)

rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf.fit(X_train_scaled, y_train)
y_pred_rf = rf.predict(X_test_scaled)

results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Decision Tree', 'Random Forest'],
    'Accuracy': [accuracy_score(y_test, y_pred_lr),
                 accuracy_score(y_test, y_pred_dt),
                 accuracy_score(y_test, y_pred_rf)],
    'F1': [f1_score(y_test, y_pred_lr),
           f1_score(y_test, y_pred_dt),
           f1_score(y_test, y_pred_rf)]
})
print(results.to_string(index=False))

## 4. Confusion Matrix and Classification Report

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, y_pred, name in zip(axes,
                             [y_pred_lr, y_pred_dt, y_pred_rf],
                             ['Logistic Reg', 'Decision Tree', 'Random Forest']):
    ConfusionMatrixDisplay.from_predictions(y_test, y_pred, ax=ax, cmap='Blues')
    ax.set_title(name)
plt.tight_layout()
plt.show()

print("\nRandom Forest Classification Report:")
print(classification_report(y_test, y_pred_rf))

## 5. ROC Curve and AUC

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

for model, name in [(lr, 'Logistic Reg'), (dt, 'Decision Tree'), (rf, 'Random Forest')]:
    if hasattr(model, 'predict_proba'):
        y_prob = model.predict_proba(X_test_scaled)[:, 1]
        fpr, tpr, _ = roc_curve(y_test, y_prob)
        auc = roc_auc_score(y_test, y_prob)
        ax.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})')

ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves')
ax.legend()
plt.show()

## 6. Cross-Validation

In [ ]:
models = [
    ('Logistic Regression', LogisticRegression(max_iter=1000, random_state=42)),
    ('Decision Tree', DecisionTreeClassifier(max_depth=5, random_state=42)),
    ('Random Forest', RandomForestClassifier(n_estimators=100, random_state=42))
]

print("5-Fold Cross-Validation (Accuracy):")
for name, model in models:
    scores = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring='accuracy')
    print(f"  {name:25s}: {scores.mean():.4f} (+/- {scores.std():.4f})")

## Try It Yourself

1. Train a classifier on imbalanced data (95/5 split). Compare accuracy vs F1. Try `class_weight='balanced'`.
2. Plot ROC curves for logistic regression, decision tree, and random forest on the same axes.
3. Use `GridSearchCV` to find the best `max_depth` for a decision tree.

In [ ]:
# Your code here